# Unified Academic Paper Classification Pipeline v7.0

This notebook provides a unified pipeline for classifying academic papers using the v7.0 models.

## Classification Hierarchy:
1. **Discipline**: Computer Science (CS), Information Systems (IS), or Information Technology (IT)
2. **Subfield**: Discipline-specific subfields
3. **Methodology**: Qualitative, Quantitative, or Mixed

## Key Features:
- Uses pre-trained v7.0 models with anti-leakage practices
- Provides confidence scores for each prediction
- Supports both single paper and batch processing
- Optional smart corrections for low-confidence predictions

---

## 📚 **Classification Categories Explained**

### 🎯 **Disciplines**

| **Discipline** | **Description** | **Focus Area** |
|-------------|-------------|-------------|
| **Computer Science (CS)** | Traditional computer science research focused on algorithms, systems, and computational theory | Technical implementation, algorithmic innovation, system design |
| **Information Systems (IS)** | Business-oriented technology research focused on organizational and managerial aspects | Business processes, organizational impact, strategic implementation |
| **Information Technology (IT)** | Infrastructure and practical technology implementation research | Network administration, system deployment, operational technology |

### 🔬 **Subfields by Discipline**

#### **Computer Science (CS) Subfields:**
| **Code** | **Full Name** | **Description** |
|----------|---------------|-----------------|
| **AI/ML** | Artificial Intelligence / Machine Learning | AI algorithms, neural networks, deep learning, intelligent systems |
| **CLOUDCS** | Cloud Computing (CS) | Distributed systems, cloud architecture, scalability algorithms |
| **CV** | Computer Vision | Image processing, pattern recognition, visual computing |
| **NLP** | Natural Language Processing | Text analysis, language models, computational linguistics |
| **SE** | Software Engineering | Software development methodologies, programming practices, code quality |
| **SEC** | Security | Cryptography, network security, cybersecurity algorithms |

#### **Information Systems (IS) Subfields:**
| **Code** | **Full Name** | **Description** |
|----------|---------------|-----------------|
| **BPM** | Business Process Management | Workflow optimization, process modeling, organizational efficiency |
| **DT** | Digital Transformation | Technology adoption, digital strategy, organizational change |
| **GOV** | IT Governance | IT strategy, compliance, risk management, digital policy |
| **HIS** | Health Information Systems | Medical informatics, electronic health records, healthcare IT |
| **KM** | Knowledge Management | Information systems, data governance, organizational learning |

#### **Information Technology (IT) Subfields:**
| **Code** | **Full Name** | **Description** |
|----------|---------------|-----------------|
| **CLOUDIT** | Cloud Computing (IT) | Cloud deployment, infrastructure as a service, cloud operations |
| **DEVOPS** | Development Operations | CI/CD, automation, infrastructure management, deployment practices |
| **EMERGING** | Emerging Technologies | IoT, blockchain, AR/VR, quantum computing applications |
| **RISK** | Risk Management | IT risk assessment, disaster recovery, business continuity |

### 📊 **Research Methodologies**

| **Methodology** | **Description** | **Typical Approaches** |
|---------------|---------------|----------------------|
| **Qualitative (QUAL)** | Non-numerical research focusing on understanding, interpretation, and meaning | Case studies, interviews, ethnography, grounded theory, content analysis |
| **Quantitative (QUANT)** | Numerical research using statistical analysis and measurable data | Experiments, surveys, statistical modeling, performance benchmarking |
| **Mixed (MIXED)** | Combination of both qualitative and quantitative approaches | Multi-method studies, triangulation, sequential explanatory designs |

---

In [27]:
# 🔧 Working Directory Setup (Run this first!)
import os
import sys

# Check current working directory
print(f"📂 Current working directory: {os.getcwd()}")

# Navigate to project root if needed
if not os.path.exists('Artefacts/current'):
    # Try to find the project root
    possible_paths = [
        '.',  # Already in project root
        '..',  # One level up
        '../..',  # Two levels up
        '../../..',  # Three levels up
        '/Users/aanandprabhu/Desktop/development/NLP-Project',  # Absolute path
    ]
    
    project_root = None
    for path in possible_paths:
        if os.path.exists(os.path.join(path, 'Artefacts/current')):
            project_root = os.path.abspath(path)
            break
    
    if project_root:
        os.chdir(project_root)
        print(f"✅ Changed to project root: {os.getcwd()}")
    else:
        print("❌ Could not find project root directory!")
        print("Please manually navigate to the NLP-Project directory")

# Verify we can see the required files
if os.path.exists('Artefacts/current'):
    print("✅ Found Artefacts/current directory")
    pkl_files = []
    for root, dirs, files in os.walk('Artefacts/current'):
        pkl_files.extend([os.path.join(root, f) for f in files if f.endswith('.pkl')])
    print(f"✅ Found {len(pkl_files)} .pkl files")
else:
    print("❌ Cannot find Artefacts/current directory")
    print("Please ensure you're running from the NLP-Project root directory")


📂 Current working directory: /Users/aanandprabhu/Desktop/development/NLP-Project
✅ Found Artefacts/current directory
✅ Found 10 .pkl files


# Unified Academic Paper Classification Pipeline v7.0

This notebook provides a unified pipeline for classifying academic papers using the v7.0 models.

## Classification Hierarchy:
1. **Discipline**: Computer Science (CS), Information Systems (IS), or Information Technology (IT)
2. **Subfield**: Discipline-specific subfields
3. **Methodology**: Qualitative, Quantitative, or Mixed

## Key Features:
- Uses pre-trained v7.0 models with anti-leakage practices
- Provides confidence scores for each prediction
- Supports both single paper and batch processing
- Optional smart corrections for low-confidence predictions

In [28]:
# Import required libraries
import joblib
import pandas as pd
import numpy as np
import json
from typing import Dict, List, Tuple, Optional, Union
import warnings
warnings.filterwarnings('ignore')

# For progress bars in batch processing
from tqdm import tqdm

In [29]:
# Define the OptimizedFeatureExtractor class (matching the original structure)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import hstack

class OptimizedFeatureExtractor(BaseEstimator, TransformerMixin):
    """Optimized feature extractor for academic paper classification."""
    
    def __init__(self):
        # Original implementation uses multiple vectorizers
        self.vectorizers = {
            'tfidf': TfidfVectorizer(
                max_features=3000,
                ngram_range=(1, 2),
                stop_words='english',
                lowercase=True,
                min_df=2,
                max_df=0.95
            )
        }
        self.is_fitted = False
    
    def fit(self, X, y=None):
        """Fit the feature extractor."""
        for name, vectorizer in self.vectorizers.items():
            vectorizer.fit(X)
        self.is_fitted = True
        return self
    
    def transform(self, X):
        """Transform texts to features."""
        if not self.is_fitted:
            raise ValueError("Feature extractor must be fitted first")
        
        # Transform with each vectorizer and concatenate
        features = []
        for name, vectorizer in self.vectorizers.items():
            features.append(vectorizer.transform(X))
        
        if len(features) == 1:
            return features[0]
        else:
            return hstack(features)
    
    def fit_transform(self, X, y=None):
        """Fit and transform in one step."""
        return self.fit(X, y).transform(X)
    
    def get_feature_names(self):
        """Get all feature names"""
        names = []
        for name, vectorizer in self.vectorizers.items():
            names.extend([f"{name}_{n}" for n in vectorizer.get_feature_names_out()])
        return names


## ⚠️ Important: Run All Setup Cells First!

**Before using the interactive classifier, make sure you run these cells in order:**

1. **Cell 1**: Import libraries
2. **Cell 2**: Define OptimizedFeatureExtractor class
3. **Cell 3**: Load models  
4. **Cell 4**: Define helper functions
5. **Cell 5**: Define classification functions
6. **Cell 6**: Fix the classification function

**Then you can run the interactive classification!**


## 1. Load Pre-trained Models

In [30]:
# Define model paths (Fixed: Uses correct "Artefacts" spelling and "current/" subdirectory)
MODEL_PATHS = {
    'discipline': 'Artefacts/current/discipline_classifier_v7/discipline_pipeline_v7.pkl',
    'cs_subfield': 'Artefacts/current/cs_subfield_classifier_v7/cs_subfield_pipeline_v7.pkl',
    'is_subfield': 'Artefacts/current/is_subfield_classifier_v7/is_subfield_pipeline_v7.pkl',
    'it_subfield': 'Artefacts/current/it_subfield_classifier_v7/it_subfield_pipeline_v7.pkl',
    'methodology': 'Artefacts/current/methodology_classifier_v7/methodology_pipeline_v7.pkl'
}

# Load all models
models = {}
for model_name, path in MODEL_PATHS.items():
    try:
        models[model_name] = joblib.load(path)
        print(f"✅ Loaded {model_name} from {path}")
    except Exception as e:
        print(f"❌ Error loading {model_name}: {e}")
        models[model_name] = None

✅ Loaded discipline from Artefacts/current/discipline_classifier_v7/discipline_pipeline_v7.pkl
✅ Loaded cs_subfield from Artefacts/current/cs_subfield_classifier_v7/cs_subfield_pipeline_v7.pkl
✅ Loaded is_subfield from Artefacts/current/is_subfield_classifier_v7/is_subfield_pipeline_v7.pkl
✅ Loaded it_subfield from Artefacts/current/it_subfield_classifier_v7/it_subfield_pipeline_v7.pkl
✅ Loaded methodology from Artefacts/current/methodology_classifier_v7/methodology_pipeline_v7.pkl


## 2. Define Classification Functions

In [31]:
def preprocess_text(text: str) -> str:
    """Clean and preprocess text input."""
    if pd.isna(text) or text is None:
        return ""
    # Convert to string and strip whitespace
    return str(text).strip()


def combine_title_abstract(title: str, abstract: str) -> str:
    """Combine title and abstract with proper formatting."""
    title = preprocess_text(title)
    abstract = preprocess_text(abstract)
    
    # Combine with space
    combined = f"{title} {abstract}".strip()
    
    # Handle edge case where both might be empty
    if not combined:
        combined = "No content available"
    
    return combined


def get_prediction_confidence(model, X) -> Tuple[str, float, np.ndarray]:
    """Get prediction with confidence score."""
    prediction = model.predict(X)[0]
    
    # Get probability scores
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X)[0]
        confidence = float(np.max(proba))
    else:
        # Fallback if model doesn't support predict_proba
        proba = np.array([1.0])
        confidence = 1.0
    
    return prediction, confidence, proba

In [32]:
def classify_single_paper(
    title: str, 
    abstract: str,
    apply_corrections: bool = True,
    confidence_threshold: float = 0.6
) -> Dict[str, Union[str, float, Dict]]:
    """
    Classify a single academic paper.
    
    Args:
        title: Paper title
        abstract: Paper abstract
        apply_corrections: Whether to apply smart corrections for low confidence
        confidence_threshold: Threshold below which to consider corrections
    
    Returns:
        Dictionary with classification results
    """
    results = {
        'input': {
            'title': title,
            'abstract': abstract[:200] + '...' if len(abstract) > 200 else abstract
        },
        'predictions': {},
        'confidence_scores': {},
        'probability_distributions': {},
        'corrections_applied': [],
        'warnings': []
    }
    
    # Combine title and abstract
    combined_text = combine_title_abstract(title, abstract)
    
    # Check for empty input
    if combined_text == "No content available":
        results['warnings'].append("Empty or missing title and abstract")
        results['predictions'] = {
            'discipline': 'Unknown',
            'subfield': 'Unknown',
            'methodology': 'Unknown'
        }
        return results
    
    # Step 1: Predict Discipline
    if models['discipline'] is not None:
        disc_pred, disc_conf, disc_proba = get_prediction_confidence(
            models['discipline'], 
            [combined_text]
        )
        results['predictions']['discipline'] = disc_pred
        results['confidence_scores']['discipline'] = disc_conf
        
        # Store probability distribution if available
        if hasattr(models['discipline'], 'classes_'):
            results['probability_distributions']['discipline'] = {
                cls: float(prob) for cls, prob in zip(models['discipline'].classes_, disc_proba)
            }
        
        # Apply correction if confidence is low
        if apply_corrections and disc_conf < confidence_threshold:
            # Simple heuristic: if very low confidence, flag for review
            results['corrections_applied'].append(
                f"Low confidence ({disc_conf:.2f}) for discipline prediction. Consider manual review."
            )
    else:
        results['predictions']['discipline'] = 'Error'
        results['warnings'].append("Discipline classifier not loaded")
        disc_pred = None
    
    # Step 2: Predict Subfield based on Discipline
    if disc_pred in ['CS', 'IS', 'IT']:
        subfield_model_name = f"{disc_pred.lower()}_subfield"
        
        if models[subfield_model_name] is not None:
            sub_pred, sub_conf, sub_proba = get_prediction_confidence(
                models[subfield_model_name], 
                [combined_text]
            )
            results['predictions']['subfield'] = sub_pred
            results['confidence_scores']['subfield'] = sub_conf
            
            # Store probability distribution
            if hasattr(models[subfield_model_name], 'classes_'):
                results['probability_distributions']['subfield'] = {
                    cls: float(prob) for cls, prob in zip(models[subfield_model_name].classes_, sub_proba)
                }
            
            # Apply correction if needed
            if apply_corrections and sub_conf < confidence_threshold:
                results['corrections_applied'].append(
                    f"Low confidence ({sub_conf:.2f}) for {disc_pred} subfield prediction."
                )
        else:
            results['predictions']['subfield'] = 'Error'
            results['warnings'].append(f"{disc_pred} subfield classifier not loaded")
    else:
        results['predictions']['subfield'] = 'Unknown'
        results['warnings'].append(f"Cannot predict subfield for discipline: {disc_pred}")
    
    # Step 3: Predict Methodology
    if models['methodology'] is not None:
        meth_pred, meth_conf, meth_proba = get_prediction_confidence(
            models['methodology'], 
            [combined_text]
        )
        results['predictions']['methodology'] = meth_pred
        results['confidence_scores']['methodology'] = meth_conf
        
        # Store probability distribution
        if hasattr(models['methodology'], 'classes_'):
            results['probability_distributions']['methodology'] = {
                cls: float(prob) for cls, prob in zip(models['methodology'].classes_, meth_proba)
            }
        
        # Apply correction if needed
        if apply_corrections and meth_conf < confidence_threshold:
            # Simple heuristic: if abstract mentions certain keywords, adjust prediction
            abstract_lower = abstract.lower()
            if 'interview' in abstract_lower or 'case study' in abstract_lower:
                if meth_pred != 'Qualitative' and meth_conf < 0.5:
                    results['predictions']['methodology'] = 'Qualitative'
                    results['corrections_applied'].append(
                        f"Corrected methodology from {meth_pred} to Qualitative based on keywords"
                    )
            elif 'regression' in abstract_lower or 'statistical' in abstract_lower:
                if meth_pred != 'Quantitative' and meth_conf < 0.5:
                    results['predictions']['methodology'] = 'Quantitative'
                    results['corrections_applied'].append(
                        f"Corrected methodology from {meth_pred} to Quantitative based on keywords"
                    )
    else:
        results['predictions']['methodology'] = 'Error'
        results['warnings'].append("Methodology classifier not loaded")
    
    return results

In [33]:
# Updated get_prediction_confidence function to work with model dictionaries

def get_prediction_confidence_fixed(model_dict, X) -> Tuple[str, float, np.ndarray]:
    """Get prediction with confidence score from model dictionary."""
    # Extract components from the dictionary
    feature_extractor = model_dict['feature_extractor']
    model = model_dict['model']
    label_encoder = model_dict['label_encoder']
    
    # Transform input to features
    X_features = feature_extractor.transform(X)
    
    # Get prediction and probabilities
    prediction_encoded = model.predict(X_features)[0]
    prediction = label_encoder.inverse_transform([prediction_encoded])[0]
    
    # Get probability scores
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_features)[0]
        confidence = float(np.max(proba))
    else:
        # Fallback if model doesn't support predict_proba
        proba = np.array([1.0])
        confidence = 1.0
    
    return prediction, confidence, proba

# Override the original function
get_prediction_confidence = get_prediction_confidence_fixed


In [34]:
# Debug: Let's check the structure of the loaded models
print("🔍 Model structure analysis:")
print(f"Discipline model keys: {list(models['discipline'].keys())}")
print(f"Feature extractor type: {type(models['discipline']['feature_extractor'])}")
print(f"Feature extractor attributes: {dir(models['discipline']['feature_extractor'])}")

# Check if it has a different attribute name
feature_extractor = models['discipline']['feature_extractor']
if hasattr(feature_extractor, 'vectorizer'):
    print("✅ Has vectorizer attribute")
elif hasattr(feature_extractor, 'tfidf_vectorizer'):
    print("✅ Has tfidf_vectorizer attribute")
elif hasattr(feature_extractor, 'transform'):
    print("✅ Has transform method - checking available attributes...")
    attrs = [attr for attr in dir(feature_extractor) if not attr.startswith('_')]
    print(f"Available attributes: {attrs}")
else:
    print("❌ No obvious vectorizer found")


🔍 Model structure analysis:
Discipline model keys: ['feature_extractor', 'model', 'label_encoder']
Feature extractor type: <class '__main__.OptimizedFeatureExtractor'>
Feature extractor attributes: ['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__sklearn_clone__', '__sklearn_tags__', '__str__', '__subclasshook__', '__weakref__', '_build_request_for_signature', '_check_feature_names', '_check_n_features', '_doc_link_module', '_doc_link_template', '_doc_link_url_param_generator', '_get_default_requests', '_get_doc_link', '_get_metadata_request', '_get_param_names', '_get_tags', '_more_tags', '_repr_html_', '_repr_html_inner', '_repr_mimebundle_', '_sklearn_auto_wrap_output_keys', '_validate_data', '_validate

In [35]:
# Alternative approach: Load artifacts directly (individual components)
print("🔄 Trying alternative loading approach...")

try:
    # Load individual artifacts instead of pipeline objects
    artifacts = {}
    for model_name in MODEL_PATHS.keys():
        artifact_path = f'Artefacts/current/{model_name}_classifier_v7/artifacts_v7.pkl'
        try:
            artifacts[model_name] = joblib.load(artifact_path)
            print(f"✅ Loaded {model_name} artifacts from {artifact_path}")
        except Exception as e:
            print(f"❌ Error loading {model_name} artifacts: {e}")
            artifacts[model_name] = None
    
    # Rebuild models dictionary with individual components
    models_rebuilt = {}
    for model_name, artifact in artifacts.items():
        if artifact is not None:
            models_rebuilt[model_name] = {
                'feature_extractor': artifact['feature_extractor'],
                'model': artifact['model'],
                'label_encoder': artifact['label_encoder']
            }
            print(f"✅ Rebuilt {model_name} model structure")
        else:
            models_rebuilt[model_name] = None
    
    # Replace the models dictionary
    models = models_rebuilt
    print("🎉 Successfully loaded all models via artifacts!")
    
except Exception as e:
    print(f"❌ Alternative loading failed: {e}")
    print("Will try another approach...")


🔄 Trying alternative loading approach...


✅ Loaded discipline artifacts from Artefacts/current/discipline_classifier_v7/artifacts_v7.pkl
✅ Loaded cs_subfield artifacts from Artefacts/current/cs_subfield_classifier_v7/artifacts_v7.pkl
✅ Loaded is_subfield artifacts from Artefacts/current/is_subfield_classifier_v7/artifacts_v7.pkl
✅ Loaded it_subfield artifacts from Artefacts/current/it_subfield_classifier_v7/artifacts_v7.pkl
✅ Loaded methodology artifacts from Artefacts/current/methodology_classifier_v7/artifacts_v7.pkl
✅ Rebuilt discipline model structure
✅ Rebuilt cs_subfield model structure
✅ Rebuilt is_subfield model structure
✅ Rebuilt it_subfield model structure
✅ Rebuilt methodology model structure
🎉 Successfully loaded all models via artifacts!


In [36]:
# Quick fix: Override the classification function to work with model dictionaries

def classify_single_paper_fixed(
    title: str, 
    abstract: str,
    apply_corrections: bool = True,
    confidence_threshold: float = 0.6
) -> Dict[str, Union[str, float, Dict]]:
    """Fixed version that works with model dictionaries."""
    
    results = {
        'input': {'title': title, 'abstract': abstract[:200] + '...' if len(abstract) > 200 else abstract},
        'predictions': {}, 'confidence_scores': {}, 'probability_distributions': {},
        'corrections_applied': [], 'warnings': []
    }
    
    combined_text = combine_title_abstract(title, abstract)
    
    if combined_text == "No content available":
        results['warnings'].append("Empty or missing title and abstract")
        results['predictions'] = {'discipline': 'Unknown', 'subfield': 'Unknown', 'methodology': 'Unknown'}
        return results
    
    # Step 1: Predict Discipline
    if models['discipline'] is not None:
        disc_pred, disc_conf, disc_proba = get_prediction_confidence(models['discipline'], [combined_text])
        results['predictions']['discipline'] = disc_pred
        results['confidence_scores']['discipline'] = disc_conf
        
        if 'label_encoder' in models['discipline']:
            classes = models['discipline']['label_encoder'].classes_
            results['probability_distributions']['discipline'] = {
                cls: float(prob) for cls, prob in zip(classes, disc_proba)
            }
    else:
        results['predictions']['discipline'] = 'Error'
        results['warnings'].append("Discipline classifier not loaded")
        disc_pred = None
    
    # Step 2: Predict Subfield
    if disc_pred in ['CS', 'IS', 'IT']:
        subfield_model_name = f"{disc_pred.lower()}_subfield"
        if models[subfield_model_name] is not None:
            sub_pred, sub_conf, sub_proba = get_prediction_confidence(models[subfield_model_name], [combined_text])
            results['predictions']['subfield'] = sub_pred
            results['confidence_scores']['subfield'] = sub_conf
            
            if 'label_encoder' in models[subfield_model_name]:
                classes = models[subfield_model_name]['label_encoder'].classes_
                results['probability_distributions']['subfield'] = {
                    cls: float(prob) for cls, prob in zip(classes, sub_proba)
                }
        else:
            results['predictions']['subfield'] = 'Error'
    else:
        results['predictions']['subfield'] = 'Unknown'
    
    # Step 3: Predict Methodology
    if models['methodology'] is not None:
        meth_pred, meth_conf, meth_proba = get_prediction_confidence(models['methodology'], [combined_text])
        results['predictions']['methodology'] = meth_pred
        results['confidence_scores']['methodology'] = meth_conf
        
        if 'label_encoder' in models['methodology']:
            classes = models['methodology']['label_encoder'].classes_
            results['probability_distributions']['methodology'] = {
                cls: float(prob) for cls, prob in zip(classes, meth_proba)
            }
    else:
        results['predictions']['methodology'] = 'Error'
    
    return results

# Override the original function
classify_single_paper = classify_single_paper_fixed


In [37]:
def classify_batch(
    df: pd.DataFrame,
    title_col: str = 'Title',
    abstract_col: str = 'Abstract',
    apply_corrections: bool = True,
    confidence_threshold: float = 0.6,
    show_progress: bool = True
) -> pd.DataFrame:
    """
    Classify multiple papers from a DataFrame.
    
    Args:
        df: DataFrame with title and abstract columns
        title_col: Name of title column
        abstract_col: Name of abstract column
        apply_corrections: Whether to apply smart corrections
        confidence_threshold: Threshold for corrections
        show_progress: Whether to show progress bar
    
    Returns:
        DataFrame with classification results
    """
    results_list = []
    
    # Setup progress bar
    iterator = tqdm(df.iterrows(), total=len(df)) if show_progress else df.iterrows()
    
    for idx, row in iterator:
        # Get title and abstract
        title = row.get(title_col, '')
        abstract = row.get(abstract_col, '')
        
        # Classify
        result = classify_single_paper(
            title, 
            abstract, 
            apply_corrections, 
            confidence_threshold
        )
        
        # Flatten results for DataFrame
        flat_result = {
            'index': idx,
            'title': title,
            'abstract': abstract[:100] + '...' if len(str(abstract)) > 100 else abstract,
            'discipline': result['predictions'].get('discipline', 'Unknown'),
            'discipline_confidence': result['confidence_scores'].get('discipline', 0),
            'subfield': result['predictions'].get('subfield', 'Unknown'),
            'subfield_confidence': result['confidence_scores'].get('subfield', 0),
            'methodology': result['predictions'].get('methodology', 'Unknown'),
            'methodology_confidence': result['confidence_scores'].get('methodology', 0),
            'corrections': '; '.join(result['corrections_applied']) if result['corrections_applied'] else 'None',
            'warnings': '; '.join(result['warnings']) if result['warnings'] else 'None'
        }
        
        results_list.append(flat_result)
    
    return pd.DataFrame(results_list)

## 3. Helper Functions for Analysis

In [38]:
# Example 1b: IT/MIXED/DEVOPS Classification
it_devops_title = "Automated Testing in Continuous Integration Pipelines: A Mixed-Methods Analysis"
it_devops_abstract = """This study evaluates automated testing practices in CI/CD pipelines through a comprehensive mixed-methods approach. We collected performance metrics from 150 production deployments across 10 organizations, measuring test execution times, failure rates, and deployment frequency. Additionally, we conducted semi-structured interviews with 30 DevOps engineers to understand their experiences with test automation challenges and best practices."""

print("🔬 Example 1b: IT/MIXED/DEVOPS Classification")
print(f"Title: {it_devops_title}")
print(f"Abstract: {it_devops_abstract[:150]}...")

result_it = classify_single_paper(it_devops_title, it_devops_abstract)
print(f"\n📊 Results:")
print(f"Discipline: {result_it['predictions']['discipline']} (confidence: {result_it['confidence_scores'].get('discipline', 0):.2f})")
print(f"Subfield: {result_it['predictions']['subfield']} (confidence: {result_it['confidence_scores'].get('subfield', 0):.2f})")
print(f"Methodology: {result_it['predictions']['methodology']} (confidence: {result_it['confidence_scores'].get('methodology', 0):.2f})")

print("\n" + "="*80)


🔬 Example 1b: IT/MIXED/DEVOPS Classification
Title: Automated Testing in Continuous Integration Pipelines: A Mixed-Methods Analysis
Abstract: This study evaluates automated testing practices in CI/CD pipelines through a comprehensive mixed-methods approach. We collected performance metrics f...

📊 Results:
Discipline: IT (confidence: 0.99)
Subfield: DEVOPS (confidence: 0.98)
Methodology: MIXED (confidence: 0.84)



In [39]:
# Example 1c: IS/QUAL/GOV Classification  
is_gov_title = "Barriers to Digital Transformation in Municipal Government: A Qualitative Investigation"
is_gov_abstract = """This qualitative study examines organizational and political barriers to digital transformation in local government. Through ethnographic observation and in-depth interviews with 45 municipal officials across three cities, we identify key challenges including legacy system dependencies, bureaucratic resistance, and citizen privacy concerns. Our grounded theory analysis reveals how institutional factors shape technology adoption decisions in public sector contexts."""

print("🏛️ Example 1c: IS/QUAL/GOV Classification")
print(f"Title: {is_gov_title}")
print(f"Abstract: {is_gov_abstract[:150]}...")

result_is = classify_single_paper(is_gov_title, is_gov_abstract)
print(f"\n📊 Results:")
print(f"Discipline: {result_is['predictions']['discipline']} (confidence: {result_is['confidence_scores'].get('discipline', 0):.2f})")
print(f"Subfield: {result_is['predictions']['subfield']} (confidence: {result_is['confidence_scores'].get('subfield', 0):.2f})")
print(f"Methodology: {result_is['predictions']['methodology']} (confidence: {result_is['confidence_scores'].get('methodology', 0):.2f})")

print("\n" + "="*80)


🏛️ Example 1c: IS/QUAL/GOV Classification
Title: Barriers to Digital Transformation in Municipal Government: A Qualitative Investigation
Abstract: This qualitative study examines organizational and political barriers to digital transformation in local government. Through ethnographic observation ...

📊 Results:
Discipline: IS (confidence: 1.00)
Subfield: GOV (confidence: 0.90)
Methodology: QUAL (confidence: 0.99)



## 🎯 **Additional Classification Examples Summary**

### **Key Differences Demonstrated:**

| **Example** | **Discipline** | **Subfield** | **Methodology** | **Why This Classification?** |
|-------------|----------------|--------------|-----------------|-------------------------------|
| **1b: DevOps Testing** | **IT** | **DEVOPS** | **MIXED** | Infrastructure focus (CI/CD) + Mixed methods (metrics + interviews) |
| **1c: Gov Digital Transform** | **IS** | **GOV** | **QUAL** | Organizational focus (transformation) + Qualitative only (ethnography + interviews) |

### **Learning Points:**

1. **IT vs IS Distinction**:
   - **IT**: Technical infrastructure, operational systems, DevOps practices
   - **IS**: Organizational transformation, business processes, strategic implementation

2. **MIXED vs QUAL Methodology**:
   - **MIXED**: Combines quantitative metrics with qualitative insights
   - **QUAL**: Purely qualitative approaches (interviews, ethnography, case studies)

3. **Subfield Specialization**:
   - **DEVOPS**: CI/CD, automation, deployment practices
   - **GOV**: Government policy, public sector IT, digital governance

These examples help users understand the classifier's decision boundaries and improve their own paper classification accuracy.


In [40]:
def analyze_confidence_distribution(results_df: pd.DataFrame) -> Dict:
    """Analyze confidence score distributions."""
    analysis = {}
    
    for level in ['discipline', 'subfield', 'methodology']:
        conf_col = f'{level}_confidence'
        if conf_col in results_df.columns:
            analysis[level] = {
                'mean': results_df[conf_col].mean(),
                'std': results_df[conf_col].std(),
                'min': results_df[conf_col].min(),
                'max': results_df[conf_col].max(),
                'low_confidence_count': (results_df[conf_col] < 0.6).sum(),
                'low_confidence_percentage': (results_df[conf_col] < 0.6).mean() * 100
            }
    
    return analysis


def export_results(results_df: pd.DataFrame, format: str = 'csv', filename: str = 'classification_results'):
    """Export results to various formats."""
    if format == 'csv':
        results_df.to_csv(f'{filename}.csv', index=False)
        print(f"✅ Results exported to {filename}.csv")
    elif format == 'json':
        results_df.to_json(f'{filename}.json', orient='records', indent=2)
        print(f"✅ Results exported to {filename}.json")
    elif format == 'excel':
        results_df.to_excel(f'{filename}.xlsx', index=False)
        print(f"✅ Results exported to {filename}.xlsx")
    else:
        print(f"❌ Unsupported format: {format}")

## 4. Example Usage - Single Paper Classification

In [41]:
# Example 1: Classify a single paper
example_title = "Deep Learning for Natural Language Processing: A Survey"
example_abstract = """This paper provides a comprehensive survey of deep learning techniques 
applied to natural language processing tasks. We review recent advances in neural 
architectures including transformers, BERT, and GPT models. Our analysis covers 
both theoretical foundations and practical applications, with empirical comparisons 
of model performance across various benchmarks."""

# Classify
result = classify_single_paper(example_title, example_abstract)

# Display results
print("\n📊 Classification Results:")
print(f"Discipline: {result['predictions']['discipline']} (confidence: {result['confidence_scores'].get('discipline', 0):.2f})")
print(f"Subfield: {result['predictions']['subfield']} (confidence: {result['confidence_scores'].get('subfield', 0):.2f})")
print(f"Methodology: {result['predictions']['methodology']} (confidence: {result['confidence_scores'].get('methodology', 0):.2f})")

if result['corrections_applied']:
    print("\n🔧 Corrections Applied:")
    for correction in result['corrections_applied']:
        print(f"  - {correction}")

if result['warnings']:
    print("\n⚠️ Warnings:")
    for warning in result['warnings']:
        print(f"  - {warning}")


📊 Classification Results:
Discipline: CS (confidence: 1.00)
Subfield: NLP (confidence: 0.98)
Methodology: QUANT (confidence: 0.83)


## 5. Example Usage - Batch Classification from CSV

In [42]:
# Example 2: Create a sample dataset for demonstration
sample_data = pd.DataFrame([
    {
        'Title': 'Machine Learning in Healthcare: A Systematic Review',
        'Abstract': 'This systematic review examines the application of machine learning techniques in healthcare settings. We analyzed 150 papers published between 2018-2023, identifying key trends and challenges in clinical decision support systems.'
    },
    {
        'Title': 'Enterprise Resource Planning Implementation: A Case Study',
        'Abstract': 'This paper presents an in-depth case study of ERP implementation in a Fortune 500 company. Through interviews with 30 stakeholders and analysis of organizational documents, we identify critical success factors.'
    },
    {
        'Title': 'Network Security in IoT Devices: Vulnerabilities and Solutions',
        'Abstract': 'We present a comprehensive analysis of security vulnerabilities in Internet of Things devices. Our study includes penetration testing of 50 commercial IoT devices and proposes a new security framework.'
    },
    {
        'Title': 'DevOps Implementation in Cloud Infrastructure: Lessons from Industry Practice',
        'Abstract': 'This paper examines DevOps practices in cloud environments through a mixed-methods study. We conducted surveys with 200 practitioners and interviewed 25 DevOps engineers across various organizations. Our findings combine quantitative performance metrics with qualitative insights from stakeholder experiences, revealing key success factors and implementation challenges.'
    },
    {
        'Title': 'Digital Government Transformation: A Qualitative Study of Policy Implementation',
        'Abstract': 'This research investigates the implementation of digital government initiatives through in-depth case studies of three municipal governments. Using ethnographic methods and semi-structured interviews with 40 government officials, we explore the organizational, political, and social factors that influence digital transformation in public sector contexts.'
    }
])

# Classify the batch
batch_results = classify_batch(sample_data, show_progress=False)

# Display results
print("\n📊 Batch Classification Results:")
display(batch_results[['title', 'discipline', 'subfield', 'methodology', 
                       'discipline_confidence', 'subfield_confidence', 'methodology_confidence']])


📊 Batch Classification Results:


,title,discipline,subfield,methodology,discipline_confidence,subfield_confidence,methodology_confidence
0,Machine Learning in Healthcare: A Systematic R...,CS,AI/ML,QUAL,0.936225,0.596866,0.822462
1,Enterprise Resource Planning Implementation: A...,IS,BPM,QUAL,0.997828,0.999306,0.544510
2,Network Security in IoT Devices: Vulnerabiliti...,CS,SEC,QUANT,0.781267,0.909611,0.617714
3,DevOps Implementation in Cloud Infrastructure:...,IT,DEVOPS,MIXED,0.946159,0.829745,0.887293
4,Digital Government Transformation: A Qualitati...,IS,GOV,QUAL,0.999332,0.956632,0.988146


### 📋 **Expected Classifications for New Examples:**

The two additional examples demonstrate different classification patterns:

#### **Example 4: DevOps Mixed Methods (IT/MIXED/DEVOPS)**
- **Title**: "DevOps Implementation in Cloud Infrastructure: Lessons from Industry Practice"
- **Expected Classification**:
  - **Discipline**: `IT` (Infrastructure focus, cloud operations)
  - **Subfield**: `DEVOPS` (Development operations, CI/CD practices)
  - **Methodology**: `MIXED` (Surveys + interviews = quantitative + qualitative)

#### **Example 5: Government IT Governance (IS/QUAL/GOV)**
- **Title**: "Digital Government Transformation: A Qualitative Study of Policy Implementation" 
- **Expected Classification**:
  - **Discipline**: `IS` (Business/organizational focus, digital transformation)
  - **Subfield**: `GOV` (IT Governance, government policy, public sector)
  - **Methodology**: `QUAL` (Ethnographic methods, case studies, interviews)

These examples showcase the classifier's ability to distinguish between IT infrastructure concerns vs. IS organizational concerns, and demonstrate mixed vs. purely qualitative methodologies.


In [43]:
# Analyze confidence distributions
confidence_analysis = analyze_confidence_distribution(batch_results)

print("\n📈 Confidence Score Analysis:")
for level, stats in confidence_analysis.items():
    print(f"\n{level.capitalize()}:")
    print(f"  - Mean confidence: {stats['mean']:.3f}")
    print(f"  - Low confidence papers: {stats['low_confidence_count']} ({stats['low_confidence_percentage']:.1f}%)")


📈 Confidence Score Analysis:

Discipline:
  - Mean confidence: 0.932
  - Low confidence papers: 0 (0.0%)

Subfield:
  - Mean confidence: 0.858
  - Low confidence papers: 1 (20.0%)

Methodology:
  - Mean confidence: 0.772
  - Low confidence papers: 1 (20.0%)


## 6. Load and Classify Real Data

In [44]:
# Load your actual CSV file
# Uncomment and modify the path below to use your own data

# input_file = 'path/to/your/data.csv'
# df = pd.read_csv(input_file)
# 
# # Classify all papers
# results = classify_batch(
#     df, 
#     title_col='Title',  # Adjust column name if needed
#     abstract_col='Abstract',  # Adjust column name if needed
#     apply_corrections=True,
#     confidence_threshold=0.6
# )
# 
# # Export results
# export_results(results, format='csv', filename='unified_pipeline_v7_results')
# export_results(results, format='json', filename='unified_pipeline_v7_results')

## 7. Interactive Single Paper Classification

In [45]:
def interactive_classify():
    """Interactive function for classifying papers via user input."""
    print("🎓 Academic Paper Classifier v7.0")
    print("="*50)
    
    # Get user input
    title = input("\nEnter paper title: ")
    print("\nEnter abstract (press Enter twice when done):")
    
    abstract_lines = []
    while True:
        line = input()
        if line == "":
            break
        abstract_lines.append(line)
    
    abstract = " ".join(abstract_lines)
    
    # Classify
    print("\n🔍 Classifying...")
    result = classify_single_paper(title, abstract)
    
    # Display results
    print("\n📊 Results:")
    print(f"\nDiscipline: {result['predictions']['discipline']}")
    print(f"  - Confidence: {result['confidence_scores'].get('discipline', 0):.2%}")
    if 'discipline' in result['probability_distributions']:
        print("  - Probabilities:")
        for cls, prob in result['probability_distributions']['discipline'].items():
            print(f"    • {cls}: {prob:.2%}")
    
    print(f"\nSubfield: {result['predictions']['subfield']}")
    print(f"  - Confidence: {result['confidence_scores'].get('subfield', 0):.2%}")
    
    print(f"\nMethodology: {result['predictions']['methodology']}")
    print(f"  - Confidence: {result['confidence_scores'].get('methodology', 0):.2%}")
    
    if result['corrections_applied']:
        print("\n🔧 Corrections:")
        for correction in result['corrections_applied']:
            print(f"  - {correction}")
    
    # Export option
    export = input("\nExport results to JSON? (y/n): ")
    if export.lower() == 'y':
        filename = input("Enter filename (without extension): ")
        with open(f"{filename}.json", 'w') as f:
            json.dump(result, f, indent=2)
        print(f"✅ Results saved to {filename}.json")

# To run interactive classification, uncomment the line below:
# interactive_classify()

## 8. Pipeline Validation on Test Data

In [46]:
# This cell demonstrates how to validate the pipeline on a holdout test set
# Uncomment and modify to use with your actual test data

# # Load test data with ground truth labels
# test_data = pd.read_csv('path/to/test_data_with_labels.csv')
# 
# # Classify
# predictions = classify_batch(test_data)
# 
# # Calculate accuracy if ground truth is available
# if 'true_discipline' in test_data.columns:
#     discipline_accuracy = (predictions['discipline'] == test_data['true_discipline']).mean()
#     print(f"Discipline Accuracy: {discipline_accuracy:.2%}")
# 
# if 'true_methodology' in test_data.columns:
#     methodology_accuracy = (predictions['methodology'] == test_data['true_methodology']).mean()
#     print(f"Methodology Accuracy: {methodology_accuracy:.2%}")
# 
# # Analyze low-confidence predictions
# low_conf_mask = predictions['discipline_confidence'] < 0.6
# print(f"\nLow confidence predictions: {low_conf_mask.sum()} ({low_conf_mask.mean():.1%})")
# 
# # Display some low-confidence examples
# if low_conf_mask.sum() > 0:
#     print("\nExamples of low-confidence predictions:")
#     display(predictions[low_conf_mask].head())

## Summary

This unified pipeline provides:

1. **Hierarchical Classification**: Discipline → Subfield → Methodology
2. **Confidence Scores**: For each prediction level
3. **Smart Corrections**: Optional corrections for low-confidence predictions
4. **Batch Processing**: Efficient classification of multiple papers
5. **Multiple Export Formats**: CSV, JSON, Excel
6. **Edge Case Handling**: Graceful handling of missing data

The pipeline uses pre-trained v7.0 models with:
- Anti-leakage practices
- Reduced feature dimensions (~3,000 features)
- XGBoost-based classifiers
- Proper train/validation/test splits

In [47]:
# Quick test - let's try a simple classification to debug
print("🧪 Testing the classifier with a simple example...")

test_title = "Machine Learning for Text Classification"
test_abstract = "This study presents a machine learning approach for automatic text classification using neural networks and deep learning techniques."

print(f"Input title: '{test_title}'")
print(f"Input abstract: '{test_abstract}'")

try:
    result = classify_single_paper(test_title, test_abstract)
    print("\n✅ Classification successful!")
    print(f"Discipline: {result['predictions']['discipline']} (confidence: {result['confidence_scores'].get('discipline', 0):.2f})")
    print(f"Subfield: {result['predictions']['subfield']} (confidence: {result['confidence_scores'].get('subfield', 0):.2f})")
    print(f"Methodology: {result['predictions']['methodology']} (confidence: {result['confidence_scores'].get('methodology', 0):.2f})")
    
    if result['warnings']:
        print(f"\n⚠️ Warnings: {result['warnings']}")
        
except Exception as e:
    print(f"❌ Error during classification: {e}")
    import traceback
    traceback.print_exc()


🧪 Testing the classifier with a simple example...
Input title: 'Machine Learning for Text Classification'
Input abstract: 'This study presents a machine learning approach for automatic text classification using neural networks and deep learning techniques.'

✅ Classification successful!
Discipline: CS (confidence: 1.00)
Subfield: AI/ML (confidence: 0.87)
Methodology: QUANT (confidence: 0.91)
